## OHLCV - Daily Prices

Extracts daily open, high, low, close, and volume (OHLCV) data from the Alpha Vantage `TIME_SERIES_DAILY` endpoint.

Two functions are defined here and intended to be called by an orchestrator notebook:
- `ohlcv_bronze(symbol, api_key)` - calls the API and returns the raw JSON payload
- `ohlcv_silver(payload)` - transforms the raw payload into a clean, typed DataFrame

> **Rate limit:** Alpha Vantage free tier allows 25 requests/day.

### Setup

Load dependencies and read the API key from the `.env` file.
Make sure you have a `.env` file at the project root with `ALPHA_VANTAGE_API_KEY=your_key_here`.

In [ ]:
import time
import requests
import pandas as pd
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv()
api_key = os.environ["ALPHA_VANTAGE_API_KEY"]

### Constants

`col_map` renames the API's verbose column names (e.g. `"1. open"`) to clean, SQL-friendly names used in the silver layer.

In [ ]:
base_url = "https://www.alphavantage.co/query"
col_map = {"1. open": "open", "2. high": "high", "3. low": "low", "4. close": "close", "5. volume": "volume"}

### Extraction Functions

The two functions below implement the **bronze/silver pattern**:
- **Bronze** stores the raw, unmodified API response. This preserves the original data so it can be reprocessed later without spending API quota.
- **Silver** transforms the raw payload into a clean, typed DataFrame ready for loading into the database.

#### Bronze

Calls the `TIME_SERIES_DAILY` endpoint and returns the raw JSON as a Python dict.
Raises a `ValueError` if the expected key is missing.

In [ ]:
def ohlcv_bronze(symbol: str, api_key: str, outputsize: str = "compact") -> dict:
    params = {"apikey": api_key, "function": "TIME_SERIES_DAILY", "symbol": symbol, "outputsize": outputsize}
    response = requests.get(base_url, params=params, timeout=30)
    response.raise_for_status()
    payload = response.json()

    if "Time Series (Daily)" not in payload:
        raise ValueError(f"Unexpected answer for {symbol}: {payload}")

    return payload

#### Silver

Receives the raw payload dict (from `ohlcv_bronze`) and returns a clean DataFrame.
Columns are renamed, types are cast, and a `symbol` column is added from the metadata.

In [ ]:
def ohlcv_silver(payload: dict) -> pd.DataFrame:
    return (
        pd.DataFrame(payload["Time Series (Daily)"]).T
        .rename(columns=col_map)
        .astype({"open": float, "high": float, "low": float, "close": float, "volume": int})
        .reset_index()
        .rename(columns={"index": "date"})
        .assign(
            symbol=payload["Meta Data"]["2. Symbol"],
            date=lambda d: pd.to_datetime(d["date"]),
        )
    )